In [1]:
import jax
import gymnax
import jax.numpy as jnp
from envs.aeroplanax import AeroPlanax

rng = jax.random.PRNGKey(0)
rng, key_reset, key_act, key_step = jax.random.split(rng, 4)

# Instantiate the environment & its settings.
env = AeroPlanax()
env_params = env.default_params()

# Reset the environment.
obs, state = env.reset(key_reset, env_params)

# Sample a random action.
action = env.action_space(env_params).sample(key_act)

# Perform the step transition.
n_obs, n_state, reward, done, _ = env.step(key_step, state, action, env_params)

In [2]:
vmap_reset = jax.jit(jax.vmap(env.reset, in_axes=(0, None)))
vmap_step = jax.jit(jax.vmap(env.step, in_axes=(0, 0, 0, None)))

num_envs = 1000000
vmap_keys = jax.random.split(rng, num_envs)

obs, state = vmap_reset(vmap_keys, env_params)
n_obs, n_state, reward, done, _ = vmap_step(vmap_keys, state, jnp.zeros((num_envs, 4)), env_params)
print(n_obs.shape)

(1000000, 16)


In [3]:
%timeit vmap_step(vmap_keys, state, jnp.zeros((num_envs, 4)), env_params)

873 ms ± 51.7 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)
